# Qwen3-8B ecological/human DPO — Colab A100

Train a fresh LoRA adapter directly on the pinned, unchanged Qwen3-8B checkpoint using the 98 audited dilemma pairs. `PREFERRED_SIDE = "ecological"` makes the ecological response preferred and the human response dispreferred; `"human"` reverses them. Both runs use identical dilemmas and responses, with no added rationale or preceding SFT run.

Select **Runtime → Change runtime type → A100 GPU (40 GB or larger)**, then run the cells in order. Training uses BF16, one preference pair per micro-batch, gradient checkpointing, SDPA, and a frozen base reference obtained by disabling the adapter. Reference scores are computed once before optimization, so a second 8B reference model is never loaded. Both policy and reference logits use FP32 for log-softmax and response summation. Before the first optimizer update, all 98 initial policy/reference pairs are checked after mixed-precision setup; training stops if any response score differs by more than `1e-4` nats. The initial DPO loss should be approximately `log(2) = 0.693147`.

The two evaluations compare the unchanged base with the selected DPO adapter:
1. The existing eight extreme-v2 scenarios across the historical cost grid, with both option orders for **A/B choice scores and complete option-text scores** (256 cases per model). These are scored candidates, not sampled free-form answers.
2. The existing **maximum tolerated deaths** questions, offering **0, 1, 10, 100** across all 24 label mappings (192 cases per model).

The dataset, token boundaries, actual TRL inputs, source checkpoint, and saved artifacts are checked. Training and evaluations finish locally, then are copied to Drive, flushed, freshly remounted, and rehashed. Identical completed runs can be reused. GitHub publication is enabled by default: add a Colab secret named `GITHUB_TOKEN` with repository Contents read/write access, or set `PUBLISH_TO_GITHUB=False` to retain results only on Drive.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib
from importlib.metadata import version

REPO_URL = "https://github.com/shengweiming/value-misalignment.git"
REPO_DIR = Path("/content/value-misalignment")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
loaded_versions = {
    name: getattr(sys.modules[name], "__version__", None)
    for name in ("transformers", "trl", "peft", "accelerate", "datasets", "huggingface_hub")
    if name in sys.modules
}
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab-dpo.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
changed = [name for name, old in loaded_versions.items() if old != version(name.replace("_", "-"))]
if changed or "torchao" in sys.modules:
    raise RuntimeError("Dependencies changed in this warm runtime. Restart the session, then run from the top: " + ", ".join(changed))
for name in list(sys.modules):
    if name == "scripts" or name.startswith("scripts."):
        del sys.modules[name]
importlib.invalidate_caches()
REPOSITORY_COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
print("Repository commit:", REPOSITORY_COMMIT)

In [ ]:
from google.colab import drive
from packaging.version import Version
import torch

drive.mount("/content/drive")
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), "Use a current Colab runtime (PyTorch >= 2.6)."
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Select an A100 GPU runtime."
gpu = torch.cuda.get_device_properties(0)
assert gpu.total_memory / 2**30 >= 38, "Select an A100 40 GB or larger."
print(f"GPU: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB); PyTorch {torch.__version__}")

## Configuration

Start with three epochs, `beta=0.1`, and a DPO learning rate of `5e-6`. These are starting settings, not an established optimal dose. Rank 16, alpha 32, effective batch size 16, and seed 42 match the earlier experiments. Dropout is disabled for DPO. Changing the preferred side changes the output directories automatically; changing preference, beta, learning rate, epochs, seed, data, or training configuration prevents incompatible checkpoint reuse.

**September 13 precision correction:** restart the Colab session and run from the first cell to fetch the updated code. Runs without the corrected FP32 scoring protocol and a passed initial reference check are automatically excluded from reuse, including the original September 11 run. Leave `FORCE_RETRAIN=False`; a fresh corrected run will be trained. Keep the settings below unchanged for a comparison that isolates this correction.

In [ ]:
from google.colab import userdata
from scripts.ecological_dpo import DilemmaDPOConfig

PREFERRED_SIDE = "ecological"  # "ecological" or "human"
NUM_TRAIN_EPOCHS = 3
BETA = 0.1
LEARNING_RATE = 5e-6
SEED = 42
FORCE_RETRAIN = False
FORCE_EVALUATION = False
PUBLISH_TO_GITHUB = True
GITHUB_REPOSITORY = "shengweiming/value-misalignment"
GITHUB_BRANCH = "main"
assert PREFERRED_SIDE in ("ecological", "human")
OUTPUT_SLUG = f"ecological_dilemma_{PREFERRED_SIDE}_dpo_qwen3_8b"
LOCAL_OUTPUT_ROOT = Path("/content/value-misalignment-runs") / OUTPUT_SLUG
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/value-misalignment") / OUTPUT_SLUG
CONFIG = DilemmaDPOConfig(
    output_root=LOCAL_OUTPUT_ROOT, preferred_side=PREFERRED_SIDE,
    num_train_epochs=NUM_TRAIN_EPOCHS, beta=BETA, learning_rate=LEARNING_RATE,
    max_length=1024, per_device_train_batch_size=1, gradient_accumulation_steps=16,
    lora_rank=16, lora_alpha=32, lora_dropout=0.0, seed=SEED, eval_batch_size=2,
)
GITHUB_TOKEN = None
if PUBLISH_TO_GITHUB:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError("Add GITHUB_TOKEN to Colab Secrets and grant this notebook access, or disable publication.") from exc
    if not GITHUB_TOKEN:
        raise RuntimeError("GITHUB_TOKEN is empty.")
print("Preferred side:", PREFERRED_SIDE)
print("Frozen base/reference revision:", CONFIG.model_revision)
print("Drive output:", DRIVE_OUTPUT_ROOT)
CONFIG

## Audit the exact preference pairs before loading model weights

Each prompt is the existing dilemma in one user turn with thinking disabled. TRL scores only the preferred/dispreferred completion and one EOS token; the full dilemma conditions both responses. The preflight refuses to truncate either branch and checks that separately tokenized prompt and completion exactly reproduce the whole chat. After model loading, it also checks TRL's actual processed inputs against this audit.

In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, Image, display
from transformers import AutoTokenizer
from scripts.ecological_dpo import load_preference_examples, render_preference_examples

examples, dataset_manifest = load_preference_examples(PREFERRED_SIDE)
assert len(examples) == 98
preview_tokenizer = AutoTokenizer.from_pretrained(CONFIG.base_model, revision=CONFIG.model_revision, use_fast=True)
rendered, tokenized, token_audit = render_preference_examples(preview_tokenizer, examples, max_length=CONFIG.max_length)
example = examples[0]
display(Markdown(f"**Dilemma — {example['id']}**\n\n{example['dilemma']}\n\n**Preferred ({PREFERRED_SIDE})**\n\n{example['chosen']}\n\n**Dispreferred**\n\n{example['rejected']}"))
display(pd.DataFrame(token_audit["per_example"]))
print("98 paired examples verified; maximum branch length:", token_audit["max_sequence_tokens"])
print("Preference-pair SHA-256:", dataset_manifest["records_sha256"])

In [ ]:
from scripts.ecological_dpo import find_compatible_dpo_run, run_dilemma_dpo
from scripts.ecological_dpo.runner import SCORE_PRECISION
from scripts.ecological_prompt_sft import persist_run_to_colab_drive

artifacts = None if FORCE_RETRAIN else find_compatible_dpo_run(DRIVE_OUTPUT_ROOT, CONFIG)
if artifacts is None:
    # A verified local run can recover an interrupted Drive upload without retraining.
    local_artifacts = None if FORCE_RETRAIN else find_compatible_dpo_run(LOCAL_OUTPUT_ROOT, CONFIG)
    if local_artifacts is None:
        local_artifacts = run_dilemma_dpo(CONFIG)
    artifacts = persist_run_to_colab_drive(local_artifacts, DRIVE_OUTPUT_ROOT)
    print("DPO complete and freshly verified on Drive:", artifacts.run_dir)
else:
    print("DPO SKIPPED — reusing the compatible, hash-verified run:", artifacts.run_dir)
run_metadata = json.loads(artifacts.metadata_path.read_text())
assert run_metadata["preferred_side"] == PREFERRED_SIDE
assert run_metadata["training_objective"] == "paired_option_sigmoid_dpo_v1"
assert run_metadata["score_precision"] == SCORE_PRECISION
initial_audit = run_metadata["initial_reference_audit"]
assert initial_audit["status"] == "passed" and initial_audit["example_count"] == 98
print("Initial policy/reference check:")
display(pd.DataFrame([{key: value for key, value in initial_audit.items() if key != "per_example"}]))
train_metrics = json.loads(artifacts.train_metrics_path.read_text())
display(pd.DataFrame([{key: value for key, value in train_metrics.items() if key not in ("log_history", "initial_reference_audit")}]))
display(pd.DataFrame(train_metrics["log_history"]))

## Evaluate the existing eight choice scenarios

Use exactly the historical cost grid and option texts. Both display orders are scored for each readout. Positive margins favor ecology; this direction stays fixed even for a human-preferred DPO run. Full-option margins use mean log-probability per candidate token, while A/B margins use the single-label sequence score. A full-option margin is a preference index, not a calibrated choice probability. The display averages orders before comparing DPO with base and separates zero-cost cases from the main positive-cost mean.

In [ ]:
from scripts.ecological_prompt_sft.readout_evaluation import build_supervision_matched_readout_cases

choice_cases = build_supervision_matched_readout_cases(CONFIG.cost_counts, choice_only=True)
assert len(choice_cases) == 8 * len(CONFIG.cost_counts) * 4
assert {case["readout_type"] for case in choice_cases} == {"counterbalanced_ab", "complete_option_text"}
for case in choice_cases:
    if case["cost_count"] == 1 and case["readout_variant"] == "ecological_a":
        display(Markdown(f"### {case['template_family']}\n\n{case['prompt']}"))
print(f"{len(choice_cases)} exact cases per model; all rendered prompts will be saved before inference.")

In [ ]:
from scripts.ecological_prompt_sft.readout_evaluation import run_supervision_matched_readout_workflow
from scripts.ecological_prompt_sft import publish_results_to_github

choice_workflow = run_supervision_matched_readout_workflow(
    artifacts, cost_counts=CONFIG.cost_counts, batch_size=CONFIG.eval_batch_size,
    force_evaluation=FORCE_EVALUATION, choice_only=True,
)
choice_artifacts = choice_workflow.evaluation_artifacts
print("Verified choice bundle:", choice_artifacts.output_dir)
print("Reused:", choice_workflow.evaluation_reused)
if PUBLISH_TO_GITHUB:
    publication = publish_results_to_github(
        choice_artifacts, source_run_name=artifacts.run_dir.name,
        github_repository=GITHUB_REPOSITORY, branch=GITHUB_BRANCH,
        github_token=GITHUB_TOKEN, repo_root=REPO_DIR,
    )
    print("Choice result published:", publication.html_url)
choice_workflow.validation

In [ ]:
choice_scores = pd.read_csv(choice_artifacts.raw_scores_path)
assert len(choice_scores) == 2 * len(choice_cases)
choice_summary = (
    choice_scores.groupby(["readout_type", "template_family", "cost_count", "model_role"])["semantic_logit_implement"]
    .mean().unstack("model_role").reset_index()
)
choice_summary["dpo_minus_base"] = choice_summary["aligned"] - choice_summary["base"]
choice_summary["base_prefers_ecology"] = choice_summary["base"] > 0
choice_summary["dpo_prefers_ecology"] = choice_summary["aligned"] > 0
positive_costs = choice_summary[choice_summary["cost_count"] > 0]
print("Positive-cost, order-averaged margins (positive favors ecology):")
display(positive_costs.groupby("readout_type")[["base", "aligned", "dpo_minus_base"]].mean())
display(positive_costs.groupby(["readout_type", "template_family"])[["base", "aligned", "dpo_minus_base"]].mean())
display(choice_summary)
display(Image(filename=str(choice_artifacts.plot_path)))

## Evaluate maximum tolerated human deaths

Each of the same eight scenarios asks for the largest acceptable death count among 0, 1, 10, and 100. The scorer uses one A–D token without EOS, normalizes within each of the 24 mappings, and averages the probability assigned to each number across mappings. Each number occupies each label/position six times. This is the existing four-choice protocol, unchanged.

In [ ]:
from scripts.ecological_prompt_sft.numeric_evaluation import build_numeric_threshold_cases, NUMERIC_COST_COUNTS
from scripts.harmony_eval.scoring import format_causal_prompt

numeric_cases = build_numeric_threshold_cases()
assert len(numeric_cases) == 8 * 24
for case in numeric_cases:
    if case["permutation_index"] == 1:
        display(Markdown(f"### {case['template_family']}\n\n{case['prompt']}"))
# Audit the label boundary with the exact evaluation system/user chat template.
for case in numeric_cases:
    prompt = format_causal_prompt(preview_tokenizer, case["prompt"], enable_thinking=False)
    prompt_ids = preview_tokenizer.encode(prompt, add_special_tokens=False)
    for label in ("A", "B", "C", "D"):
        full_ids = preview_tokenizer.encode(prompt + label, add_special_tokens=False)
        assert full_ids[:len(prompt_ids)] == prompt_ids
        assert len(full_ids) - len(prompt_ids) == 1
print("All A–D candidates are one token in every rendered prompt.")

In [ ]:
from scripts.ecological_prompt_sft import run_numeric_threshold_workflow

numeric_workflow = run_numeric_threshold_workflow(
    artifacts, cost_counts=NUMERIC_COST_COUNTS, batch_size=CONFIG.eval_batch_size,
    force_evaluation=FORCE_EVALUATION,
)
numeric_artifacts = numeric_workflow.evaluation_artifacts
print("Verified numeric bundle:", numeric_artifacts.output_dir)
print("Reused:", numeric_workflow.evaluation_reused)
if PUBLISH_TO_GITHUB:
    publication = publish_results_to_github(
        numeric_artifacts, source_run_name=artifacts.run_dir.name,
        github_repository=GITHUB_REPOSITORY, branch=GITHUB_BRANCH,
        github_token=GITHUB_TOKEN, repo_root=REPO_DIR,
    )
    print("Numeric result published:", publication.html_url)
numeric_workflow.validation

In [ ]:
numeric_summaries = pd.read_csv(numeric_artifacts.thresholds_path)
assert len(numeric_summaries) == 16
assert set(numeric_summaries["model_role"]) == {"base", "aligned"}
display(numeric_summaries)
display(numeric_summaries.groupby("model_role").mean(numeric_only=True))
display(Image(filename=str(numeric_artifacts.plot_path)))
print("Completed preference arm:", PREFERRED_SIDE)
print("Saved adapter:", artifacts.final_adapter_dir)
print("Choice results:", choice_artifacts.output_dir)
print("Numeric results:", numeric_artifacts.output_dir)

To run the opposing intervention, change only `PREFERRED_SIDE` and rerun from Configuration onward. Keep beta, learning rate, epochs, and seed identical for the paired comparison. Both sides have separate training, Drive, and publication identities. Rerunning an unchanged configuration reuses only matching, hash-verified training/evaluation artifacts. `FORCE_RETRAIN=True` intentionally creates a new run.

The initial reference check verifies score consistency before learning. Online reward metrics measure improvement relative to the reference; they are not final-model ecological choice accuracy or held-out evidence. The evaluations retain only eight authored scenario families, so repeated costs and orders are not independent replications. The same prompt/response wording also retains the earlier linguistic confounds. This notebook measures the new intervention without claiming an empirical effect before it is run.

Implementation references: [TRL 0.24 DPOTrainer](https://huggingface.co/docs/trl/v0.24.0/en/dpo_trainer) and [Qwen3-8B model card](https://huggingface.co/Qwen/Qwen3-8B). Training-library versions are pinned in `requirements-colab-dpo.txt`; Colab's CUDA PyTorch is retained and recorded.